# Generate Teams API Data

This notebook generates the unified `teams.json` for the `/teams` API endpoint.

**Input:** `mens_teams.csv`, `womens_teams.csv`

**Output:** `teams.json` (to be uploaded to Azure Blob Storage)

In [ ]:
import pandas as pd
import json
import os

## Load Source Data

In [ ]:
mens_path = os.path.abspath('../../data/mens_teams.csv')
womens_path = os.path.abspath('../../data/womens_teams.csv')

mens_df = pd.read_csv(mens_path)
womens_df = pd.read_csv(womens_path)

print(f"Men's teams: {len(mens_df)}")
print(f"Women's teams: {len(womens_df)}")

In [ ]:
# Preview the data
mens_df.head()

## Merge into Unified Dataset

We'll use SR key as the primary identifier and merge the two datasets, tracking which programs each school has.

In [ ]:
# Add program flags
mens_df['has_mens_program'] = True
womens_df['has_womens_program'] = True

# Get sets of SR keys
mens_keys = set(mens_df['SR key'].dropna())
womens_keys = set(womens_df['SR key'].dropna())

# Find overlaps and unique
both = mens_keys & womens_keys
mens_only = mens_keys - womens_keys
womens_only = womens_keys - mens_keys

print(f"Both programs: {len(both)}")
print(f"Men's only: {len(mens_only)}")
print(f"Women's only: {len(womens_only)}")
print(f"Total unique: {len(both) + len(mens_only) + len(womens_only)}")

In [ ]:
# Merge: use mens as base, then add womens-only schools
# For schools in both, prefer mens data (they should be identical anyway)

merged_df = mens_df.copy()
merged_df['has_womens_program'] = merged_df['SR key'].isin(womens_keys)

# Add women's only schools
womens_only_df = womens_df[womens_df['SR key'].isin(womens_only)].copy()
womens_only_df['has_mens_program'] = False

merged_df = pd.concat([merged_df, womens_only_df], ignore_index=True)

print(f"Total merged teams: {len(merged_df)}")

In [ ]:
# Verify merge
merged_df[['School', 'SR key', 'has_mens_program', 'has_womens_program']].head(20)

## Transform to API Format

Convert to snake_case field names and handle null values.

In [ ]:
def transform_team(row):
    """Transform a row to API format."""
    return {
        'key': row['SR key'] if pd.notna(row['SR key']) else None,
        'school': row['School'] if pd.notna(row['School']) else None,
        'name': row['NCAA Name'] if pd.notna(row.get('NCAA Name')) else None,
        'location': row['City, State'] if pd.notna(row['City, State']) else None,
        'ncaa_key': row['NCAA key'] if pd.notna(row.get('NCAA key')) else None,
        'color': row['background-color'] if pd.notna(row.get('background-color')) else None,
        'has_mens_program': bool(row.get('has_mens_program', False)),
        'has_womens_program': bool(row.get('has_womens_program', False)),
    }

teams_list = [transform_team(row) for _, row in merged_df.iterrows()]

# Filter out any teams without a key
teams_list = [t for t in teams_list if t['key']]

# Sort alphabetically by key
teams_list = sorted(teams_list, key=lambda x: x['key'])

print(f"Total teams in output: {len(teams_list)}")

In [ ]:
# Preview transformed data
teams_list[:5]

In [ ]:
# Check UConn specifically
[t for t in teams_list if t['key'] == 'connecticut']

## Validate Data Quality

In [ ]:
# Check for issues
no_ncaa_key = [t for t in teams_list if t['ncaa_key'] is None]
no_color = [t for t in teams_list if t['color'] is None]
no_name = [t for t in teams_list if t['name'] is None]

print(f"Teams without NCAA key: {len(no_ncaa_key)}")
print(f"Teams without color: {len(no_color)}")
print(f"Teams without full name: {len(no_name)}")

# Check for duplicates
keys = [t['key'] for t in teams_list]
duplicates = [k for k in keys if keys.count(k) > 1]
print(f"Duplicate keys: {set(duplicates) if duplicates else 'None'}")

## Save Output

In [ ]:
# Save to JSON file
output_path = os.path.abspath('teams.json')

with open(output_path, 'w') as f:
    json.dump(teams_list, f, indent=2)

print(f"Saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")

## Upload to Azure Blob Storage

Run this cell to upload the teams.json to the mlmb-api container.

In [ ]:
# Uncomment and run to upload to Azure
# Requires: pip install azure-storage-blob
# Requires: AZURE_STORAGE_CONNECTION_STRING environment variable

# from azure.storage.blob import BlobServiceClient
# import os

# connection_string = os.environ.get('AZURE_STORAGE_CONNECTION_STRING')
# blob_service = BlobServiceClient.from_connection_string(connection_string)
# container_client = blob_service.get_container_client('mlmb-api')

# with open('teams.json', 'rb') as f:
#     container_client.upload_blob('teams', f, overwrite=True)

# print('Uploaded teams.json to mlmb-api/teams')